# Exploring Chignolin with AlphaFold 3 and BioEmu

## Data Source

This repository contains processed structure predictions generated using AlphaFold 3. For more information, see [google-deepmind/alphafold3](https://github.com/google-deepmind/alphafold3/tree/main) and [Abramson, J. et al. Nature (2024)](https://www.nature.com/articles/s41586-024-07487-w).

This repository contains processed structure predictions generated using BioEMU. For more information, see [microsoft/bioemu](https://github.com/microsoft/bioemu) and [Lewis, S. et al. Science (2025)](https://www.science.org/doi/10.1126/science.adv9817).

## AlphaFold 3

In [7]:
import nglview as nv
import MDAnalysis as mda
import matplotlib.pyplot as plt

# 1. Define paths to the files you just saved
gro_path = "./alphafold3/aligned/chignolin_topology.gro"
xtc_path = "./alphafold3/aligned/chignolin_ensemble.xtc"

# 2. Load the universe
u = mda.Universe(gro_path, xtc_path)
view = nv.NGLWidget()

# 3. Setup colors based on total frames in the loaded trajectory
n_frames = len(u.trajectory)
cmap = plt.get_cmap("hsv", n_frames)

# 4. Loop through frames and add as separate components
for i in range(n_frames):
    # Set the universe to the specific frame
    u.trajectory[i]
    
    # Snapshot current coordinates into a static component
    # We create a temporary universe from the current positions
    comp = view.add_component(mda.Universe(gro_path, u.atoms.positions))
    comp.clear_representations()
    
    # Coloring logic
    color = cmap(i)
    hex_color = '#%02x%02x%02x' % tuple(int(255*x) for x in color[:3])
    comp.add_cartoon(color=hex_color)

# 5. Final touches
view.add_cartoon(opacity=0.3)
view.center()
view.parameters = {"backgroundColor": "white"}
view

NGLWidget()

In [8]:
# Show movie

import nglview as nv
import MDAnalysis as mda
import os

# 1. Define your GROMACS-generated paths
gro_path = "./alphafold3/aligned/chignolin_topology.gro"
xtc_path = "./alphafold3/aligned/chignolin_ensemble.xtc"

if os.path.exists(gro_path) and os.path.exists(xtc_path):
    # 2. Load the Universe
    # MDA automatically detects the GROMACS formats
    u = mda.Universe(gro_path, xtc_path)

    # 3. Create the view using the MDAnalysis adapter
    view = nv.show_mdanalysis(u)

    # 4. Cleanup & Setup
    view.clear()
    view.parameters = {"backgroundColor": "white"}

    # 5. Protein Representation (Rainbow Spectrum)
    view.add_representation("cartoon", 
                            selection="protein", 
                            color_scheme="residueindex")

    # 6. Show the structural "cloud" (Sidechains)
    # Since Chignolin is a hairpin, seeing the sidechains helps 
    # visualize the stability of the hydrogen bonding.
    view.add_representation("licorice", 
                            selection="protein", 
                            opacity=0.3, 
                            scale=0.5)

    # 7. Player Controls
    view.player.parameters = {
        "delay": 100, 
        "step": 1, 
        "mode": "loop"
    }

    view.center("protein")
    display(view)
else:
    print("Files not found. Please check your aligned directory.")

NGLWidget(max_frame=19)

## BioEmu

In [9]:
import nglview as nv
import MDAnalysis as mda
import matplotlib.pyplot as plt
import os
from IPython.display import display

# 1. Paths
# pdb_path = './bioemu/chignolin/chignolin_full_atom/samples_md_equil.pdb'

top_path = './bioemu/chignolin/aligned/aligned_topology.gro'
xtc_path = './bioemu/chignolin/aligned/aligned_mdanalysis.xtc'

u = mda.Universe(top_path, xtc_path)
view = nv.NGLWidget()

# 2. Match your original logic: Divide the hsv map by the number of frames
step = 1 # Show every frame
frames = range(0, len(u.trajectory), step)
cmap = plt.get_cmap("hsv", len(frames)) 

# 3. Loop and add components (Mimicking the 'path' logic)
for i, frame_idx in enumerate(frames):
    u.trajectory[frame_idx] 
    
    # Snapshot the coordinates into a static component
    comp = view.add_component(mda.Universe(top_path, u.atoms.positions))
    
    # Convert RGBA → hex (Directly from your first script)
    color = cmap(i)
    hex_color = '#%02x%02x%02x' % tuple(int(255*x) for x in color[:3])
    
    comp.clear_representations()
    comp.add_cartoon(color=hex_color) # No individual opacity here, matching Script 1

# 4. Match the "Final Touch" from Script 1
# This adds the semi-transparent "ghost" layer over the entire cloud
view.add_cartoon(opacity=0.5)

view.center()
view.parameters = {"backgroundColor": "white"}
display(view)

NGLWidget()

In [10]:
# Show movie for BioEmu samples

import nglview as nv
import MDAnalysis as mda
import os
from IPython.display import display

# 1. Define paths to your BioEmu output
# We use the PDB as the topology and the original XTC for frames
# pdb_path = './bioemu/chignolin/chignolin_full_atom/samples_md_equil.pdb'
top_path = './bioemu/chignolin/aligned/aligned_topology.gro'
xtc_path = './bioemu/chignolin/aligned/aligned_mdanalysis.xtc'

if os.path.exists(top_path) and os.path.exists(xtc_path):
    # 2. Load the Universe
    u = mda.Universe(top_path, xtc_path)

    # 3. Create the view
    view = nv.show_mdanalysis(u)

    # 4. Cleanup & Setup
    view.clear()
    view.parameters = {"backgroundColor": "white"}

    # 5. Protein Representation (Rainbow Spectrum)
    # Blue at N-term (Gly1), Red at C-term (Gly10)
    view.add_representation("cartoon", 
                            selection="protein", 
                            color_scheme="residueindex")

    # 6. Show the structural "cloud" (Sidechains)
    view.add_representation("licorice", 
                            selection="protein", 
                            opacity=0.3, 
                            scale=0.5)

    # 7. Player Controls
    view.player.parameters = {
        "delay": 100, 
        "step": 1, 
        "mode": "loop"
    }

    view.center("protein")
    display(view)
else:
    print(f"Files not found.\nPDB exists: {os.path.exists(top_path)}\nXTC exists: {os.path.exists(xtc_path)}")

NGLWidget(max_frame=17)

# Potential energy of models

In [11]:
import os
import numpy as np
import mdtraj as md
from openmm import app, unit, LangevinIntegrator
from openmm.app import ForceField, Simulation, Modeller

# Define your paths as requested
top_path_af = "./alphafold3/aligned/chignolin_topology.gro"
xtc_path_af = "./alphafold3/aligned/chignolin_ensemble.xtc"

top_path_bioemu = './bioemu/chignolin/aligned/aligned_topology.gro'
xtc_path_bioemu = './bioemu/chignolin/aligned/aligned_mdanalysis.xtc'

def get_energies(top_path, xtc_path):
    # 1. Load trajectory
    traj = md.load(xtc_path, top=top_path)
    
    # 2. Prepare the Topology once
    base_topology = traj.topology.to_openmm()
    forcefield = ForceField('amber99sbildn.xml', 'tip3p.xml')
    
    # Create a template modeller to get the "Full Atom" topology (adding H)
    template_modeller = Modeller(base_topology, traj.xyz[0]*unit.nanometers)
    template_modeller.addHydrogens(forcefield)
    full_topology = template_modeller.getTopology()
    
    # 3. Setup System and Simulation
    system = forcefield.createSystem(full_topology, nonbondedMethod=app.NoCutoff)
    integrator = LangevinIntegrator(300*unit.kelvin, 1/unit.picosecond, 0.002*unit.picoseconds)
    simulation = Simulation(full_topology, system, integrator)
    
    energies = []
    print(f"Calculating energies for {len(traj)} frames...")
    
    # 4. Loop through frames
    for i in range(len(traj)):
        # Map heavy atoms to full topology and re-add hydrogens for this frame
        frame_modeller = Modeller(base_topology, traj.xyz[i]*unit.nanometers)
        frame_modeller.addHydrogens(forcefield)
        
        # Set positions in the context
        simulation.context.setPositions(frame_modeller.getPositions())
        
        # Minimize to fix 'guessed' hydrogen clashes
        simulation.minimizeEnergy(maxIterations=10) 
        
        state = simulation.context.getState(getEnergy=True)
        energy = state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
        energies.append(energy)
        
        if (i+1) % 50 == 0:
            print(f" Processed {i+1}/{len(traj)} frames...")
            
    return np.array(energies)

# --- Execution ---
print("Scoring AlphaFold3 Ensemble...")
energies_af = get_energies(top_path_af, xtc_path_af)

print("\nScoring BioEmu Ensemble...")
energies_bio = get_energies(top_path_bioemu, xtc_path_bioemu)

print("\nAll calculations finished successfully.")

Scoring AlphaFold3 Ensemble...
Calculating energies for 20 frames...

Scoring BioEmu Ensemble...
Calculating energies for 18 frames...

All calculations finished successfully.


In [12]:
import pandas as pd

def display_formatted_ensemble(energies, label):
    # 1. Create and Sort
    df = pd.DataFrame({
        'Frame': range(len(energies)),
        f'Energy_{label} (kJ/mol)': energies
    })
    df_sorted = df.sort_values(by=f'Energy_{label} (kJ/mol)').reset_index(drop=True)
    
    # --- MODIFICATION: Shift index to start at 1 instead of 0 ---
    df_sorted.index = range(1, len(df_sorted) + 1)
    
    # 2. Transpose for the two-row look
    df_t = df_sorted.T
    
    # 3. Apply Styling (Precision 0 for Frames, Precision 1 for Energy)
    styled_df = df_t.style.format(precision=0, subset=pd.IndexSlice[[df_t.index[0]], :]) \
                          .format(precision=1, subset=pd.IndexSlice[[df_t.index[1]], :])
    
    print(f"--- {label} Ensemble (Sorted: Lowest to Largest) ---")
    display(styled_df)

# Execute
display_formatted_ensemble(energies_af, "AF3")
print("\n")
display_formatted_ensemble(energies_bio, "BioEmu")

--- AF3 Ensemble (Sorted: Lowest to Largest) ---


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
Frame,19,1,5,9,14,10,11,7,16,0,4,12,8,2,18,13,3,17,6,15
Energy_AF3 (kJ/mol),-300.4,-226.4,-217.8,-206.5,-189.6,-186.4,-182.3,-164.1,-157.0,-153.4,-148.9,-142.1,-140.6,-139.9,-136.5,-118.1,-113.2,-106.8,-88.0,-44.4




--- BioEmu Ensemble (Sorted: Lowest to Largest) ---


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
Frame,9,14,11,12,13,6,3,10,1,8,16,15,17,5,4,0,2,7
Energy_BioEmu (kJ/mol),-418.4,-334.3,-235.8,-204.3,-189.9,-181.7,-141.0,-139.2,-120.6,-107.5,-97.0,-82.8,-82.7,-81.7,-70.1,-28.5,13.5,206.0
